# Kepler Exoplanet Candidate Classification

# 0: Loading and inspecting the Dataset

In this part, we load the NASA Kepler Objects of Interest (KOI) Cumulative Table that we downloaded and put in the "data" subfolder.
We inspect the dataset's dimensions and display the first 3 rows to see what features we have.

In [29]:
import pandas as pd
import numpy as np

# Load the dataset from the data folder
df_raw = pd.read_csv('data/cumulative.csv', comment='#')

#dimensions (Rows and Columns)
print(f"Dataset Shape: {df_raw.shape}")

#Display first 3 rows
display(df_raw.head(3))

Dataset Shape: (9564, 140)


,kepid,kepoi_name,kepler_name,koi_disposition,koi_vet_stat,koi_vet_date,koi_pdisposition,koi_score,koi_fpflag_nt,koi_fpflag_ss,...,koi_dicco_mdec,koi_dicco_mdec_err,koi_dicco_msky,koi_dicco_msky_err,koi_dikco_mra,koi_dikco_mra_err,koi_dikco_mdec,koi_dikco_mdec_err,koi_dikco_msky,koi_dikco_msky_err
0,10797460,K00752.01,Kepler-227 b,CONFIRMED,Done,2018-08-16,CANDIDATE,1.000,0,0,...,0.200,0.16,0.200,0.170,0.080,0.130,0.310,0.170,0.320,0.160
1,10797460,K00752.02,Kepler-227 c,CONFIRMED,Done,2018-08-16,CANDIDATE,0.969,0,0,...,0.000,0.48,0.390,0.360,0.490,0.340,0.120,0.730,0.500,0.450
2,10811496,K00753.01,NaN,CANDIDATE,Done,2018-08-16,CANDIDATE,0.000,0,0,...,-0.034,0.07,0.042,0.072,0.002,0.071,-0.027,0.074,0.027,0.074


In [30]:
#Counting how many rows belong to each category
print("Original Target Distribution:")
print(df_raw['koi_disposition'].value_counts(dropna=False))

print("\nUnique stars (kepid):", df_raw["kepid"].nunique(), "of", len(df_raw), "rows")
print("Duplicate KOI names:", df_raw["kepoi_name"].duplicated().sum())

#Grouping the data by each individual star
stars = df_raw.groupby("kepid").size()
print("KOIs per star:")
#counts how many candidate objects/planets belong to that specific star
print(stars.value_counts().sort_index())

Original Target Distribution:
koi_disposition
FALSE POSITIVE    4839
CONFIRMED         2748
CANDIDATE         1977
Name: count, dtype: int64

Unique stars (kepid): 8214 of 9564 rows
Duplicate KOI names: 0
KOIs per star:
1    7276
2     661
3     179
4      68
5      25
6       3
7       2
Name: count, dtype: int64


Checking data types and missing values

In [31]:
num_cols= df_raw.select_dtypes(include=np.number).columns
obj_cols= df_raw.select_dtypes(exclude=np.number).columns

print(f"There is {len(num_cols)} numeric columns, and {len(obj_cols)} non-numeric columns")
print("Non-numeric:", list(obj_cols))

#calculate the percentage of missing values per col:
missing= (df_raw.isna().mean()*100).sort_values(ascending=False)
print(missing)


There is 123 numeric columns, and 17 non-numeric columns
Non-numeric: ['kepoi_name', 'kepler_name', 'koi_disposition', 'koi_vet_stat', 'koi_vet_date', 'koi_pdisposition', 'koi_disp_prov', 'koi_comment', 'koi_fittype', 'koi_limbdark_mod', 'koi_parm_prov', 'koi_tce_delivname', 'koi_quarters', 'koi_trans_mod', 'koi_datalink_dvr', 'koi_datalink_dvs', 'koi_sparprov']
koi_sma_err2      100.0
koi_sma_err1      100.0
koi_teq_err1      100.0
koi_teq_err2      100.0
koi_longp_err2    100.0
                  ...  
koi_time0bk         0.0
koi_time0           0.0
koi_disp_prov       0.0
dec                 0.0
ra                  0.0
Length: 140, dtype: float64


# 1:Target definition

In the dataset there is a third category which is "CANDIDATE" which means that object has not been fully verified by NASA, so we do not know which class it belongs to. We exclude it from training and testing.

- CONFIRMED = 1 (it is a planet)
- FALSE POSITIVE = 0 (not a planet)

In [32]:
LABEL = "koi_disposition"

# Keep only confirmed planets and false positives
df = df_raw[df_raw[LABEL].isin(["CONFIRMED", "FALSE POSITIVE"])].copy()
df["y"] = (df[LABEL] == "CONFIRMED").astype(int)
groups = df["kepid"].values

print("Labeled rows:", len(df), "\nExcluded CANDIDATE rows:", (df_raw[LABEL] == "CANDIDATE").sum())
print()
print(df["y"].value_counts().rename({1: "CONFIRMED", 0: "FALSE POSITIVE"}))
print(f"Positive rate: {df['y'].mean():.3f}")

Labeled rows: 7587 
Excluded CANDIDATE rows: 1977

y
FALSE POSITIVE    4839
CONFIRMED         2748
Name: count, dtype: int64
Positive rate: 0.362


# 2:Data Leakage removal & feature exclusions

 **Leaky Feature Elimination:**
   - **koi_score and koi_pdisposition:** these are scored calculated after the final verdict is determined. If the model sees them, it is like they are seing the answer
   - **kepid, kepoi_name:** These are identifiers
   - **koi_fpflag_nt, koi_fpflag_ss, koi_fpflag_co, koi_fpflag_ec:** They are the exact diagnostic tests scientists use to categorize a false positive. If we leave them, it means that the model is copying human decisions instead of learning patterns
   - **koi_vet_stat, koi_vet_date, koi_disp_prov, koi_commen koi_tce_delivname:** These are human administrative notes and timestamps. They don't contain any physical properties about the planet/star
   - **kepler_name:** Only confirmed planets have names in this column. So if a row has a "kepler_name", the model knows immediately that it belongs to class 1


   




In [33]:
LABEL = "koi_disposition"

LEAKY= ["koi_pdisposition", "koi_score", "kepler_name",
    "koi_fpflag_nt", "koi_fpflag_ss", "koi_fpflag_co", "koi_fpflag_ec",
    "koi_vet_stat", "koi_vet_date", "koi_disp_prov", "koi_comment",
    "koi_tce_delivname"]

IDS= ["kepid", "kepoi_name", "rowid"]


# 3: Train-Test Split

Multiple KOIs can orbit the exact same star and share the same star characteristics. So we need to group our split by kepid to ensure that all planet candidates that orbit the same star stay together on one sie(either all of them go to training or all to testing) so we can prevent data leakage and force our model to learn the general transit phyics instead of simply memorizing the properties of a specific star

In [34]:
from sklearn.model_selection import StratifiedGroupKFold

# Initialize StratifiedGroupKFold to get a single deterministic 80/20 train/test split
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

train_idx, test_idx = next(sgkf.split(df, df["y"], groups=df["kepid"]))

# final train and test dfs
df_train = df.iloc[train_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

print(f"Train: {len(df_train)} rows, positive rate {df_train['y'].mean():.3f}")
print(f"Test : {len(df_test)} rows, positive rate {df_test['y'].mean():.3f}")
# Verify that no star appears in both sets
assert set(df_train["kepid"]).isdisjoint(set(df_test["kepid"]))

print("No stars shared between training and test sets.")

Train: 6068 rows, positive rate 0.362
Test : 1519 rows, positive rate 0.362
No stars shared between training and test sets.


# 4:Feature filtering(training data only)

**Data Filtering:** We automatically drop features that are 100% missing (`NaN`) or contain zero variance (constant values in the training rows), because they do not carry useful predictive information during training. We determine these columns using training data only, then use the same selected columns for testing.

We also verify whether the time columns are redundant before dropping the redundant versions.

In [35]:
#drop_always contains the IDs and the target labels + LEAKY features
drop_always = set(IDS + [LABEL, "y"]+ LEAKY)
#numeric values except IDs and target labels
numeric_all = []
for col in df_train.select_dtypes(include="number").columns:
  if col not in drop_always:
    numeric_all.append(col)

# remove the columns whose values are all missing(100% empty)
all_nan = []
clean_features = []
for col in numeric_all:
  if df_train[col].isna().all():
    all_nan.append(col)
  else:
    clean_features.append(col)

# remove the columns who have only 1 value ()
const = []
final_features = []
for col in clean_features:
  if df_train[col].nunique(dropna=True) <= 1:
    const.append(col)
  else:
    final_features.append(col)
clean_features = final_features

# Verify redundancy using training data only
redundant_time_cols = []

# Check the transit epoch offset
time_diff = (
    df_train["koi_time0"] - df_train["koi_time0bk"]
)

valid_diff = time_diff.dropna()

if (
    len(valid_diff) > 0
    and np.allclose(
        valid_diff,
        2454833.0,
        rtol=0,
        atol=1e-3
    )
    and df_train["koi_time0"].isna().equals(
        df_train["koi_time0bk"].isna()
    )
):
    redundant_time_cols.append("koi_time0")

print(
    "Maximum offset error:",
    (valid_diff - 2454833.0).abs().max()
)

# Check the two uncertainty columns
for s in ["_err1", "_err2"]:
    if np.allclose(
        df_train["koi_time0" + s],
        df_train["koi_time0bk" + s],
        rtol=0,
        atol=1e-4,
        equal_nan=True
    ):
        redundant_time_cols.append("koi_time0" + s)

clean_features = [
    c for c in clean_features
    if c not in redundant_time_cols
]

print("Removed redundant time columns:", redundant_time_cols)
print(f"\nClean features remaining after dropping redundant time cols: {len(clean_features)}")

print(f"Clean features remaining: {len(clean_features)}")
print("Dropped all-NaN columns:", all_nan)
print("Dropped constant columns:", const)

Maximum offset error: 0.0005000000819563866
Removed redundant time columns: ['koi_time0', 'koi_time0_err1', 'koi_time0_err2']

Clean features remaining after dropping redundant time cols: 92
Clean features remaining: 92
Dropped all-NaN columns: ['koi_eccen_err1', 'koi_eccen_err2', 'koi_longp', 'koi_longp_err1', 'koi_longp_err2', 'koi_ingress', 'koi_ingress_err1', 'koi_ingress_err2', 'koi_sma_err1', 'koi_sma_err2', 'koi_incl_err1', 'koi_incl_err2', 'koi_teq_err1', 'koi_teq_err2', 'koi_model_dof', 'koi_model_chisq', 'koi_sage', 'koi_sage_err1', 'koi_sage_err2']
Dropped constant columns: ['koi_eccen', 'koi_ldm_coeff4', 'koi_ldm_coeff3']


# 5: Examine missingness, distributions, and skewness(training data only)

Before fixing missing values or transforming skewed features, we inspect the selected training features. Missing-value percentages show which measurements are incomplete. `describe()` displays count, mean, standard deviation, minimum, maximum, and the 25%, 50%, and 75% quartiles. Skewness shows whether a distribution has a longer tail on one side. These statistics are computed on the training data only.

In [36]:
# Inspect the selected training features before transforming them
X_tr = df_train[clean_features]

# Percentage of missing values in each training feature
missing_pct = (X_tr.isna().mean() * 100).sort_values(ascending=False)
print("Missing values in training features (%):")
display(missing_pct[missing_pct > 0].to_frame("Missing (%)"))

# Descriptive statistics: count, mean, standard deviation, quartiles and range
print("Training feature descriptive statistics:")
display(X_tr.describe().T)

# Skewness: positive values indicate a longer right tail
skew = X_tr.skew()
print("Training feature skewness (highest first):")
display(skew.sort_values(ascending=False).to_frame("Skewness"))


Missing values in training features (%):


,Missing (%)
koi_fwm_stat_sig,12.755438
koi_bin_oedp_sig,11.799604
koi_fwm_prao_err,9.739618
koi_fwm_prao,9.739618
koi_fwm_pdeco_err,9.492419
...,...
koi_gmag,0.510877
koi_kmag,0.329598
koi_jmag,0.329598
koi_hmag,0.329598


Training feature descriptive statistics:


,count,mean,std,min,25%,50%,75%,max
koi_period,6068.0,52.135997,111.753606,2.418425e-01,2.303351,8.258993,31.581213,1.071233e+03
koi_period_err1,5798.0,0.001825,0.008266,1.100000e-08,0.000004,0.000022,0.000147,1.725000e-01
koi_period_err2,5798.0,-0.001825,0.008266,-1.725000e-01,-0.000147,-0.000022,-0.000004,-1.100000e-08
koi_time0bk,6068.0,162.544574,63.547894,1.205159e+02,132.617572,136.511620,169.972328,1.472522e+03
koi_time0bk_err1,5798.0,0.008208,0.020888,8.700000e-06,0.000885,0.003040,0.007707,5.690000e-01
...,...,...,...,...,...,...,...,...
koi_dikco_mra_err,5695.0,0.355285,0.608355,6.700000e-02,0.076000,0.180000,0.460000,3.300000e+01
koi_dikco_mdec,5695.0,-0.073509,2.840336,-7.660000e+01,-0.340000,-0.016000,0.270000,3.120000e+01
koi_dikco_mdec_err,5695.0,0.363069,0.527972,6.700000e-02,0.078000,0.200000,0.490000,2.200000e+01
koi_dikco_msky,5695.0,1.974021,3.298719,0.000000e+00,0.180000,0.500000,2.300000,8.960000e+01


Training feature skewness (highest first):


,Skewness
koi_depth_err1,74.334400
koi_prad,53.686448
koi_fwm_prao_err,47.390979
koi_fwm_pdeco_err,46.270625
koi_prad_err1,43.019138
...,...
koi_fwm_prao,-32.292843
koi_srad_err2,-33.077667
koi_insol_err2,-44.530484
koi_prad_err2,-53.186123


In [37]:
# Select strictly positive, strongly right-skewed features
# using training data only
strictly_pos = [
    c for c in clean_features
    if (
        X_tr[c].notna().any()
        and (X_tr[c].dropna() > 0).all()
    )
]

log_cols = [
    c for c in strictly_pos
    if skew[c] > 2
]

print("Features selected for log transformation:", len(log_cols))
print(log_cols)

Features selected for log transformation: 30
['koi_period', 'koi_period_err1', 'koi_time0bk', 'koi_time0bk_err1', 'koi_duration', 'koi_depth', 'koi_ror', 'koi_srho', 'koi_prad', 'koi_sma', 'koi_teq', 'koi_insol', 'koi_dor', 'koi_max_sngle_ev', 'koi_max_mult_ev', 'koi_count', 'koi_tce_plnt_num', 'koi_srad', 'koi_fwm_sra_err', 'koi_fwm_sdec_err', 'koi_fwm_srao_err', 'koi_fwm_sdeco_err', 'koi_fwm_prao_err', 'koi_fwm_pdeco_err', 'koi_dicco_mra_err', 'koi_dicco_mdec_err', 'koi_dicco_msky_err', 'koi_dikco_mra_err', 'koi_dikco_mdec_err', 'koi_dikco_msky_err']


# 6. Preprocessing Pipelines

Different machine learning models have different preprocessing requirements.

**Logistic Regression:**
Missing values: We use median imputation to replace missing values because Logistic Regression cannot handle NaN values directly. The median is less sensitive to extreme values than the mean.

Missing-value indicators: We add binary indicators to preserve information about which observations originally contained missing values.

Feature scaling: We use StandardScaler to standardize features, helping numerical optimization and making regularization more comparable across features.

We apply a signed logarithmic transformation to strongly right-skewed features identified using training data, reducing the influence of extreme values before imputation and standardization.

**XGBoost:**
XGBoost can handle missing numerical values internally and does not require feature scaling. Therefore, we use the original numerical features without imputation or standardization.


All preprocessing operations that learn statistics from data are fitted within the model pipeline. This prevents preprocessing leakage during cross-validation.

In [38]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, FunctionTransformer

# Prepare input features and target labels
X_train = df_train[clean_features].copy()
X_test = df_test[clean_features].copy()

y_train = df_train["y"]
y_test = df_test["y"]

# Apply log transformation to selected columns
def apply_log(X, columns):
    X = X.copy()

    for col in columns:
        # Signed log1p: identical to log1p for positive values,
        # but also handles unexpected negative values safely.
        X[col] = np.sign(X[col]) * np.log1p(np.abs(X[col]))

    return X

# Logistic Regression preprocessing
logistic_preprocessor = Pipeline(steps=[
    ("log", FunctionTransformer(
        apply_log,
        kw_args={"columns":log_cols},
        validate=False
    )),
    ("imputer", SimpleImputer(
        strategy="median",
        add_indicator=True
    )),
    ("scaler", StandardScaler())
])

# XGBoost uses X_train and X_test directly,
# retaining missing values and original feature scales.

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("Log-transformed features:", len(log_cols))
print("Training missing entries:",
      int(X_train.isna().sum().sum()))

print("Logistic preprocessing defined (not fitted yet).")
print("XGBoost inputs retain NaN and original feature scales.")

X_train: (6068, 92)
X_test: (1519, 92)
Log-transformed features: 30
Training missing entries: 24532
Logistic preprocessing defined (not fitted yet).
XGBoost inputs retain NaN and original feature scales.
